# CS180 Project 3 — Part A: Fun with Flow Models

Complete sampling and editing experiments using PixNerd and the supplied personal photographs, web image, and drawings. No model training is performed.


## Run in Google Colab
1. Choose **Runtime → Change runtime type → GPU**. Use a high-memory GPU/runtime if available; the supplied model and text encoder run in float32.
2. Choose **Runtime → Run all**. All five personal inputs and the Campanile are embedded in this notebook: no image uploads are required.
3. If setup reports that packages changed, restart the session once and run all again.
4. The last cell downloads `parta_results.zip`, containing your images, measured results, and a standalone Part A webpage. Save this notebook **with outputs** using File → Download → Download .ipynb.

`GROUP = "all"` runs every CS180 Part A exercise. If needed, select `core`, `editing`, or `illusions` in the settings cell and run the notebook in separate passes. Outputs accumulate in `results/partA/` during the same runtime. Completed individual edits/samples are reused, and each Euler update is checkpointed so an interrupted job can resume in that runtime. Use the last cell at any time to download partial progress. A Colab runtime reset deletes its local files; download progress before disconnecting.

The CS280A-only extra transformations and course logo are omitted. The generated webpage describes the implementation and reports measured errors; inspect the generated samples and add your own visual observations before submission.


In [ ]:
#@title Setup (run once)
# Supplied support code and images are embedded: no separate files are needed.
import os, sys, time, io, base64, zipfile, hashlib
from pathlib import Path
_cs180_started = time.perf_counter()
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
PART = 'parta'
_work = Path('/content/cs180-hw3') if IN_COLAB else (Path('/kaggle/working') if IN_KAGGLE else Path.cwd())
_work.mkdir(parents=True,exist_ok=True)
_support = base64.b64decode('...')
assert hashlib.sha256(_support).hexdigest() == '4b8c000ba0d28853cf3c39f9642c03c248cc9255389a0817de970aa71fbc9b05', 'Damaged notebook support data'
with zipfile.ZipFile(io.BytesIO(_support)) as _archive:
    for _entry in _archive.infolist():
        _target=(_work/_entry.filename).resolve()
        if _work.resolve() not in _target.parents: raise ValueError('Unsafe embedded path')
    _archive.extractall(_work)
os.chdir(_work)
if str(_work) not in sys.path: sys.path.insert(0,str(_work))
from colab_setup import configure
configure(PART,install_dependencies=IN_COLAB or IN_KAGGLE)
print('Setup complete. Continue with the notebook below.')


In [ ]:
import os, json, time, math
from pathlib import Path
from uuid import uuid4
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from torchvision.transforms.functional import gaussian_blur
from flow_model import FlowModel, preprocess, to_pil, MODEL_ID, MODEL_REVISION, TEXT_REVISION

torch.set_num_threads(4)
ROOT = Path.cwd()
GROUP = "all"  # @param ["all", "core", "editing", "illusions"]
if GROUP not in ("all", "core", "editing", "illusions"):
    raise ValueError("FLOW_GROUP must be all, core, editing, or illusions.")
# A new folder for this run keeps earlier results available.
OUT = ROOT / "results" / "partA"
OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINTS = OUT / "checkpoints"
CHECKPOINTS.mkdir(exist_ok=True)
SEED, H, W = 180, 0.02, 7
torch.manual_seed(SEED)
_model_key = (MODEL_REVISION, TEXT_REVISION, 'cuda:0', 'float32')
if globals().get('_loaded_model_key') != _model_key or 'model' not in globals():
    model = FlowModel()
    _loaded_model_key = _model_key
else:
    print('Reusing the already-loaded PixNerd model in this kernel.')
predict_velocity = model.predict_velocity
device = model.device
condition = model.encode("a high quality photo")
null_condition = model.encode("")


def load_input(path):
    if not str(path).strip():
        raise ValueError("Set the image path to a file you uploaded or saved in this runtime.")
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}. Check the path in the file browser.")
    return preprocess(path).to(device)


def noise(seed):
    return torch.randn((1, 3, 512, 512), generator=torch.Generator().manual_seed(seed)).to(device)


def panel(name, images, columns=4, title=None):
    items = list(images.items())
    fig, axs = plt.subplots(math.ceil(len(items) / columns), columns,
                            figsize=(4 * columns, 4 * math.ceil(len(items) / columns)),
                            squeeze=False, layout="constrained")
    if title is not None:
        fig.suptitle(title, wrap=True)
    for ax in axs.flat:
        ax.axis("off")
    for ax, (label, value) in zip(axs.flat, items):
        ax.imshow(to_pil(value))
        ax.set_title(label, fontsize=9, wrap=True)
    fig.savefig(OUT / (name + ".png"), dpi=110)
    plt.show()
    plt.close(fig)


def save(name, x):
    to_pil(x).save(OUT / (name + ".png"))


def guided(x, t, c=condition, w=W):
    return cfg_velocity(x, t, c, w, predict_velocity, null_condition)


metrics = json.loads((OUT / "metrics.json").read_text()) if (OUT / "metrics.json").exists() else {}
metrics.update({"group": GROUP, "seed": SEED, "step_size": H, "cfg_w": W,
           "model": MODEL_ID, "revision": MODEL_REVISION, "text_revision": TEXT_REVISION,
           "gpu": torch.cuda.get_device_name(0), "visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
           "torch": torch.__version__})
print("Results folder:", OUT)
print(metrics)


def save_metrics():
    (OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))


def resumable_sample(name, x_start, t_start, velocity, cache_key=""):
    # The fingerprint prevents reuse after changing prompts, seeds, or settings.
    """Checkpoint every Euler update; preserve completed final tensors by experiment name."""
    fingerprint = hashlib.sha256(json.dumps([name, SEED, H, W, MODEL_REVISION, cache_key]).encode()).hexdigest()[:16]
    path = CHECKPOINTS / (name + "_" + fingerprint + ".pt")
    start = t_start
    x = x_start
    if path.exists():
        state = torch.load(path, map_location=device, weights_only=True)
        x, start = state["x"], state["t"]
        if start >= 1:
            print("Reusing", name, flush=True)
            return x
        print(f"Resuming {name} at t={start:.2f}", flush=True)
    steps = math.ceil(round((1 - start) / H, 9))
    with torch.inference_mode():
        for k in range(steps):
            t = start + k * H
            h = min(H, 1 - t)
            x = x + h * velocity(x, t)
            next_t = 1.0 if k == steps - 1 else start + (k + 1) * H
            temp = path.with_suffix(".tmp")
            torch.save({"x": x.detach().cpu(), "t": next_t}, temp)
            temp.replace(path)
            if (k + 1) % 10 == 0 or next_t == 1:
                print(f"{name}: t={next_t:.2f}", flush=True)
    return x


def edit_cached(name, source, start, prompt):
    initial = interpolate(source, noise(SEED), start)
    return resumable_sample(name, initial, start,
        lambda x, t: cfg_velocity(x, t, prompt, W, predict_velocity, null_condition),
        cache_key=hashlib.sha256(prompt.detach().float().cpu().numpy().tobytes()).hexdigest() if isinstance(prompt, torch.Tensor) else prompt)


## Sampling implementations
The functions below implement the required interpolation, Euler integration, CFG, editing, visual anagrams, and frequency-combined sampling.


In [ ]:
# Part A sampling functions.
import math


def interpolate(image, noise, t):
    # x_t = (1 - t) * eps + t * x_1   (t = 0: pure noise, t = 1: clean image)
    return (1 - t) * noise + t * image


def one_step_denoise(x, t, condition, predict_velocity):
    # Jump straight to t = 1 along the current velocity: x_1_hat = x_t + (1 - t) * v
    v = predict_velocity(x, t, condition)
    return x + (1 - t) * v


@torch.inference_mode()
def euler_sample(x_start, t_start, step_size, velocity):
    """Integrate dx/dt = velocity(x, t) from t_start to exactly t = 1.

    Returns (final_x, states), where states is a list of (t, x) pairs holding the
    initial state, every fifth update, and the final state.
    """
    # Number of updates; rounding guards against float error such as 0.5/0.02 = 25.000000000000004.
    n_steps = math.ceil(round((1 - t_start) / step_size, 9))
    x = x_start
    states = [(float(t_start), x)]
    for k in range(n_steps):
        t = t_start + k * step_size              # computed directly, so error doesn't accumulate
        last = k == n_steps - 1
        h = (1 - t) if last else step_size       # shorten the final step to land exactly on t = 1
        x = x + h * velocity(x, t)               # recompute velocity at the current state every step
        t_next = 1.0 if last else t_start + (k + 1) * step_size
        if (k + 1) % 5 == 0 or last:
            states.append((t_next, x))
    return x, states


def cfg_velocity(x, t, condition, w, predict_velocity, null_condition=''):
    # v_cfg = v_c + w * (v_c - v_u); both branches use the same x and t.
    v_c = predict_velocity(x, t, condition)
    if w == 0:
        return v_c  # w = 0 is plain conditional prediction; skip the unconditional pass
    v_u = predict_velocity(x, t, null_condition)
    return v_c + w * (v_c - v_u)


def edit(image, noise, t_start, condition, predict_velocity, w=7, step_size=0.02):
    # SDEdit: noise the image to t_start, then flow back to t = 1 with CFG.
    x = interpolate(image, noise, t_start)
    v = lambda x, t: cfg_velocity(x, t, condition, w, predict_velocity, '')
    out, _ = euler_sample(x, t_start, step_size, v)
    return out


# PROVIDED: rotate by 180 degrees, keeping the batch and RGB axes unchanged.
def rotate180(x):
    return torch.flip(x, (-2, -1))


def visual_anagrams(noise, condition_a, condition_b, predict_velocity, w=7, step_size=0.02):
    # Average the upright velocity for prompt a with the rotated-view velocity for prompt b
    # (rotated back so both live in the upright frame).
    def v(x, t):
        v1 = cfg_velocity(x, t, condition_a, w, predict_velocity, '')
        v2 = rotate180(cfg_velocity(rotate180(x), t, condition_b, w, predict_velocity, ''))
        return (v1 + v2) / 2
    out, _ = euler_sample(noise, 0, step_size, v)
    return out


# PROVIDED: at 512 x 512, this uses kernel size 265 and sigma 16.
def lowpass(x, sigma=16.0):
    kernel = 2 * round(16.5 * x.shape[-1] / 64) + 1
    return gaussian_blur(x, [kernel, kernel], [sigma, sigma])


def make_hybrids(noise, condition_a, condition_b, predict_velocity, w=7, step_size=0.02):
    # Low frequencies from prompt a (seen from far away), high frequencies from prompt b (up close).
    def v(x, t):
        v_a = cfg_velocity(x, t, condition_a, w, predict_velocity, '')
        v_b = cfg_velocity(x, t, condition_b, w, predict_velocity, '')
        return lowpass(v_a) + (v_b - lowpass(v_b))
    out, _ = euler_sample(noise, 0, step_size, v)
    return out


# 1. Visualizing Training and Sampling

Use the linear flow-matching convention from the webpage:
$x_t=(1-t)\epsilon+t x_1$, where $\epsilon\sim\mathcal{N}(0,I)$.
**Time 0 is noise; time 1 is a clean image.** The path velocity target is $x_1-\epsilon$.
We use the pretrained model; no model training is required in Part A.

### Choose your image for Sections 1.1–1.5

Use **one image of your own, not the provided image of the Campanile**, throughout these five exercises.
Upload or add your image to the runtime, then set `YOUR_IMAGE_PATH` below to its path.
In Colab, upload through the Files sidebar and copy the uploaded file's path.
In Kaggle, add the image as an input and copy its path from the file browser.

The supplied preprocessing center-crops and resizes the image to $512\times512$ and
normalizes RGB to $[-1,1]$. The Campanile will be loaded separately for Sections 3.1–3.2.

In [ ]:
# All five inputs were unpacked from the embedded support bundle by setup.
INPUTS = {
    "own1": ROOT / "images/own/own1.jpg",
    "own2": ROOT / "images/own/own2.jpg",
    "web": ROOT / "images/web/web.jpg",
    "drawing1": ROOT / "images/drawings/drawing1.jpg",
    "drawing2": ROOT / "images/drawings/drawing2.jpg",
}
def upload(name):
    path = INPUTS[name]
    if not path.is_file():
        raise FileNotFoundError(f"Missing embedded image: {path}; rerun setup.")
    return str(path)
YOUR_IMAGE_PATH = upload("own1")
image = load_input(YOUR_IMAGE_PATH)
assert image.shape == (1, 3, 512, 512) and image.dtype == torch.float32
metrics["inputs"] = {name: str(path.relative_to(ROOT)) for name, path in INPUTS.items()}
metrics["section_1_image"] = "Brick building facade in warm sunlight"
save_metrics()
display(to_pil(image))


## 1.1 Constructing Noisy Training Inputs

Implement `interpolate(image, noise, t)` using $x_t=(1-t)\epsilon+t x_1$.
Sample one noise tensor and reuse it at $t=0.25,0.50,0.75$.
Show your original image and the three noisy inputs, labeled by time.

**Hints:** Check both endpoints. Keep `image`, `epsilon`, and `noisy` for Sections 1.2–1.5;
the original image/noise pair is needed to compute the ground-truth velocity in Section 1.4.

In [ ]:
epsilon = noise(SEED)
noisy = {t: interpolate(image, epsilon, t) for t in [.25,.50,.75]}
assert torch.equal(interpolate(image,epsilon,0),epsilon)
assert torch.equal(interpolate(image,epsilon,1),image)
if GROUP in ("all","core"):
    torch.save({"image": image.cpu(), "epsilon": epsilon.cpu(),
                "noisy": {t: x.cpu() for t, x in noisy.items()}}, OUT / "noisy_inputs.pt")
    panel("training_inputs", {"original": image, **{f"t={t}":x for t,x in noisy.items()}})

## 1.2 Classical Denoising

For each saved input, restore the clean-image amplitude with x/t, then apply a Gaussian filter.
Try different blur strengths and show your best result beside each noisy input.
The provided cell sweeps sigma=1,2,4,8,16 and selects lowest mean squared error to the known clean image;
this is a baseline comparison, not a learned model or a guarantee of perceptual quality.

In [ ]:
if GROUP in ("all","core"):
    blurred = {}
    for t,x in noisy.items():
        candidates = {s: gaussian_blur(x/t, [int(6*s+1)]*2, [s,s]) for s in [1.,2.,4.,8.,16.]}
        sigma = min(candidates, key=lambda s: ((candidates[s]-image)**2).mean().item())
        blurred[t] = candidates[sigma]
        panel(f"blur_{t}", {"noisy":x,f"blur sigma={sigma}":blurred[t]}, columns=2)
    metrics["blur_selection"] = "Lowest full-image MSE among sigma=1,2,4,8,16, measured before clipping"
    metrics["blur_sigma"] = {
        str(t): min([1.,2.,4.,8.,16.], key=lambda s: float(((gaussian_blur(x/t, [int(6*s+1)]*2, [s,s])-image)**2).mean()))
        for t,x in noisy.items()
    }
    save_metrics()


## 1.3 One-Step Denoising

Implement `one_step_denoise`: x̂₁=xₜ+(1−t)vθ(xₜ,t,c).
Use exactly one prediction for each of the same three noisy tensors, with “a high quality photo”.
Display original, input, and estimate. Check velocity shape, dtype, and device.

In [ ]:
if GROUP in ("all","core"):
    v = predict_velocity(noisy[.5], .5, condition)
    assert v.shape == image.shape and v.dtype == image.dtype and v.device == image.device
    assert torch.isfinite(v).all()
    one_step = {t:one_step_denoise(x,t,condition,predict_velocity) for t,x in noisy.items()}
    for t,x in noisy.items():
        panel(f"one_step_{t}", {"original":image,"noisy":x,"one step":one_step[t]}, columns=3)

## 1.4 Visualizing Velocity

In Section 1.3, we added $(1-t)\hat v_t$ to the noisy image to estimate the clean image.
Here, visualize what the model adds and compare it with the update $x_1-x_t$
that would recover your original image.

Use the same image and saved noisy inputs at $t=0.25,0.50,0.75$, with
`"a high quality photo"`.

**Deliverables**
- Show the original image once.
- At each time, show the noisy input, predicted update $(1-t)\hat v_t$,
  ground-truth update $(1-t)v_{\mathrm{gt}}$, and an error heatmap, labeled by time.
  You may arrange the displays however you like.

**Implementation hints**
- Predict $\hat v_t=v_\theta(x_t,t,c)$. Using the saved noise, compute
  $v_{\mathrm{gt}}=x_1-\epsilon$, so $(1-t)v_{\mathrm{gt}}=x_1-x_t$.
- For the error map, use RMS across RGB **at each pixel**:
  $E_t=\sqrt{\operatorname{mean}_{\mathrm{RGB}}([(1-t)(\hat v_t-v_{\mathrm{gt}})]^2)}$.
  Square the channel differences, average over RGB, and take the square root before display scaling.
- Use one shared symmetric RGB scale for all updates, with zero at middle gray.
- Use `matplotlib.pyplot.imshow` with a linear scale for errors. Choose `vmin` and `vmax`
  that work for your image, keeping them the same at all three times.

Complete the calculation function below. The following cell supplies an example display.

In [ ]:
@torch.inference_mode()
def velocity_diagnostics(x_t, t, clean_image, epsilon, condition, predict_velocity):
    """Return predicted_update, ground_truth_update, and error_map.

    Both updates have shape (B, 3, H, W).
    The error map has shape (B, H, W): average only over RGB, keeping each pixel.
    Return raw values before display scaling or clipping.
    """
    v_hat = predict_velocity(x_t, t, condition)        # model's velocity at (x_t, t)
    v_gt = clean_image - epsilon                       # true path velocity x_1 - eps
    predicted_update = (1 - t) * v_hat
    ground_truth_update = (1 - t) * v_gt               # equals clean_image - x_t
    error_map = ((predicted_update - ground_truth_update) ** 2).mean(dim=1).sqrt()  # RMS over RGB
    return predicted_update, ground_truth_update, error_map


In [ ]:
# PROVIDED display helpers. Adjust the error limits or layout for your results.
ERROR_VMIN = 0.0
ERROR_VMAX = None  # None uses the maximum across all three maps; you may choose a smaller limit.


def signed_rgb(update, limit):
    # Map the shared signed range [-limit, limit] to RGB [0, 1].
    value = update.detach().float().cpu()[0].permute(1, 2, 0)
    return (0.5 + value / (2 * limit)).numpy()


if GROUP in ("all", "core"):
    velocity_results = {}
    for t in (0.25, 0.50, 0.75):
        predicted_update, target_update, error_map = velocity_diagnostics(
            noisy[t], t, image, epsilon, condition, predict_velocity)
        if predicted_update.shape != image.shape or target_update.shape != image.shape:
            raise ValueError("Each update should have the same shape as the input image.")
        if error_map.shape != (image.shape[0], *image.shape[-2:]):
            raise ValueError("The error map should have shape (B, H, W); average only over RGB.")
        velocity_results[t] = tuple(value.detach().float().cpu() for value in
                                    (predicted_update, target_update, error_map))

    signed_limit = max(1e-8, *(float(value.abs().max())
                             for predicted, target, _ in velocity_results.values()
                             for value in (predicted, target)))
    error_max = ERROR_VMAX
    if error_max is None:
        error_max = max(1e-8, *(float(error.max()) for _, _, error in velocity_results.values()))
    if error_max <= ERROR_VMIN:
        raise ValueError("Choose ERROR_VMAX greater than ERROR_VMIN.")

    print("Original image")
    display(to_pil(image))
    save("velocity_original", image)
    for t, (predicted_update, target_update, error_map) in velocity_results.items():
        fig, axes = plt.subplots(1, 4, figsize=(16, 4), layout="constrained")
        fig.suptitle(f"t = {t:.2f}")
        axes[0].imshow(to_pil(noisy[t]))
        axes[1].imshow(signed_rgb(predicted_update, signed_limit))
        axes[2].imshow(signed_rgb(target_update, signed_limit))
        axes[3].imshow(error_map[0].numpy(), vmin=ERROR_VMIN, vmax=error_max)
        for ax, title in zip(axes, ("Noisy image", r"Predicted $(1-t)\hat v$",
                                    r"Ground truth $(1-t)v_{\mathrm{gt}}$", "Error (RGB RMS)")):
            ax.set_title(title)
            ax.axis("off")
        fig.colorbar(axes[3].images[0], ax=axes[3], fraction=0.046, label="RGB RMS error")
        fig.savefig(OUT / f"velocity_{t:.2f}.png", dpi=110)
        plt.show()
        plt.close(fig)
    metrics["velocity_signed_limit"] = signed_limit
    metrics["velocity_error_limits"] = [ERROR_VMIN, error_max]
    metrics["velocity_rms_mean"] = {str(t): float(error.mean()) for t, (_, _, error) in velocity_results.items()}
    save_metrics()


## 1.5 Iterative Denoising with Euler Sampling

Implement `euler_sample(x_start,t_start,step_size,velocity)`:
xₜ₊ₕ=xₜ+h·v(xₜ,t). Recompute velocity each step and shorten the final step to reach 1 exactly.
Return the final tensor and saved states. Denoise the saved t=0.50 input with h=0.02 (25 updates).
Show the initial state, every fifth update, and final result, labeled by time. Compare blur, one-step,
and iterative results on this same input.

In [ ]:
if GROUP in ("all","core"):
    calls=[]
    def constant_v(x,t):
        calls.append(t); return torch.ones_like(x)
    toy=torch.zeros(1,3,4,4,device=device)
    for start,step,expected in [(0,.02,50),(.5,.02,25),(.93,.02,4)]:
        calls.clear(); end,states=euler_sample(toy,start,step,constant_v)
        assert len(calls)==expected and states[-1][0]==1
        assert torch.allclose(end,torch.full_like(toy,1-start),atol=1e-6)
    iterative,trajectory=euler_sample(noisy[.5],.5,H,lambda x,t:predict_velocity(x,t,condition))
    panel("trajectory",{f"t={t:.2f}":x for t,x in trajectory},3)
    panel("comparison",{"original":image,"blur":blurred[.5],"one step":one_step[.5],"Euler":iterative})
    metrics["reconstruction_mse"]={k:float(((v-image)**2).mean()) for k,v in
        {"blur":blurred[.5],"one_step":one_step[.5],"euler":iterative}.items()}
    metrics["euler_checks"]="50 full steps, 25 half steps, shortened final step passed"
    save_metrics()


## 1.6 Generating Images from Noise

Generate five independent Gaussian initial states and integrate from 0 to 1 (50 Euler updates).
Use “a high quality photo”. Show all five results and record the seeds.

In [ ]:
if GROUP in ("all","core"):
    samples={}
    for i in range(5):
        x=resumable_sample(f"conditional_{i}", noise(SEED+i), 0, lambda x,t:predict_velocity(x,t,condition))
        samples[f"seed={SEED+i}"]=x; save(f"conditional_{i}",x)
    panel("conditional_samples",samples)


# 2. Conditional Generation and Guidance

Section 1 used a fixed text prompt. Here, implement classifier-free guidance to strengthen
the prompt's effect and compare the resulting samples.

## 2.1 Classifier-Free Guidance

Implement v_cfg=v_c+w(v_c−v_u), with empty-string null condition.
Both branches must evaluate the same x and t. w=0 gives ordinary conditional prediction.
Generate five images at w=7 and h=0.02. Compare to Section 1.6 using the same initial noises.


In [ ]:
if GROUP in ("all","core"):
    x=noise(SEED)
    assert torch.equal(cfg_velocity(x,.5,condition,0,predict_velocity,null_condition),predict_velocity(x,.5,condition))
    cfg_samples={}
    for i in range(5):
        x=resumable_sample(f"cfg_{i}", noise(SEED+i), 0, guided)
        cfg_samples[f"seed={SEED+i}, w={W}"]=x; save(f"cfg_{i}",x)
    panel("cfg_samples",cfg_samples)
    metrics["cfg_zero_check"]=True
    save_metrics()


# 3. Creative Applications

Use Euler step size `h=0.02`, CFG weight `w=7`, and the empty-string prompt for the
unconditional prediction throughout this section.

## 3.1 Image-to-Image Editing

Implement `edit`: construct $(1-t_{\mathrm{start}})\epsilon+t_{\mathrm{start}}x_1$
and integrate from the starting time to 1 with CFG.
Use `"a high quality photo"` and the **seven** starting times
**0.02, 0.04, 0.06, 0.10, 0.14, 0.25, 0.50**.

Show the original and all seven edits for the **Campanile and two images of your own**.
The Campanile is included in setup. Set the two image paths below; you may reuse your image
from Section 1.1 as one of them. These inputs will also be used in Section 3.2.

In [ ]:
# The same seven times are used throughout Sections 3.1 and 3.2.
STARTS = [0.02, 0.04, 0.06, 0.10, 0.14, 0.25, 0.50]
TEST_IMAGE_PATHS = {
    "my_image_1": YOUR_IMAGE_PATH,
    "my_image_2": upload("own2"),  # Second image of your own.
}

if GROUP in ("all", "editing"):
    # Load Campanile separately; `image` remains your Section 1 image.
    campanile = load_input(ROOT / "assets/campanile.jpg")
    edit_inputs = {"campanile": campanile,
                   **{name: load_input(path) for name, path in TEST_IMAGE_PATHS.items()}}
    for name, y in edit_inputs.items():
        panel("edit_" + name, {"original": y, **{
            f"t={t:.2f}": edit_cached(f"edit_{name}_{t:.2f}", y, t, condition)
            for t in STARTS}})
        print("Edited", name, flush=True)


### Editing Hand-Drawn and Web Images

Repeat the same procedure and **seven starting times** for **one image from the web and
two of your own drawings**. Download or draw the images, upload/add them to the runtime,
and set their paths below. The preprocessing helper center-crops to a square and resizes
to 512×512, so keep your drawings centered.

In [ ]:
WEB_IMAGE_PATH = upload("web")  # Image downloaded from the web.
DRAWING_PATHS = {
    "drawing_1": upload("drawing1"),  # Your first drawing.
    "drawing_2": upload("drawing2"),  # Your second drawing.
}

if GROUP in ("all", "editing"):
    creative_inputs = {"web_image": load_input(WEB_IMAGE_PATH),
                       **{name: load_input(path) for name, path in DRAWING_PATHS.items()}}
    for name, y in creative_inputs.items():
        panel("edit_" + name, {"original": y, **{
            f"t={t:.2f}": edit_cached(f"edit_{name}_{t:.2f}", y, t, condition)
            for t in STARTS}})
        print("Edited", name, flush=True)


## 3.2 Text-Guided Image Editing

Reuse the Campanile and two personal images from Section 3.1. For each image, choose a prompt
describing your desired edit and replace the generic photo prompt with its embedding.
Use the same **seven starting times: 0.02, 0.04, 0.06, 0.10, 0.14, 0.25, 0.50**.
Show the original and all edits, with the chosen prompt and times.

Larger starting times preserve more of the original image. Smaller starting times add more
noise and give the model more room to change the image according to the prompt.

In [ ]:
EDIT_PROMPTS = {
    "campanile": "a photograph of a lighthouse on a rocky coast",
    "my_image_1": "a photograph of an old brick apartment building covered in flowering green vines, warm afternoon sunlight",
    "my_image_2": "a photograph of a red leather notebook with a black elastic strap resting on a canvas bag on a car seat",
}

if GROUP in ("all", "editing"):
    if set(EDIT_PROMPTS) != set(edit_inputs):
        raise ValueError("Provide one editing prompt for each image from Section 3.1.")
    if any(not prompt.strip() for prompt in EDIT_PROMPTS.values()):
        raise ValueError("Fill in all three EDIT_PROMPTS before running this cell.")
    for name, prompt in EDIT_PROMPTS.items():
        c = model.encode(prompt)
        y = edit_inputs[name]
        panel("text_edit_" + name, {"original": y, **{
            f"t={t:.2f}": edit_cached(f"text_edit_{name}_{t:.2f}", y, t, c)
            for t in STARTS}}, title=f'{name}: "{prompt}"')
        print("Text edited", name, flush=True)
    metrics["edit_prompts"] = EDIT_PROMPTS
    save_metrics()


## 3.3 Visual Anagrams

Let F rotate an image 180 degrees (flip both spatial axes, not just vertical reflection).
Average v_cfg(x,t,c_a) and F(v_cfg(F(x),t,c_b)), and integrate using the same Euler sampler.
Choose two prompt pairs and generate two illusions. Show each upright and rotated with both prompts. Several prompt/seed attempts
may be needed: a successful numerical run does not imply both subjects are recognizable.

In [ ]:
if GROUP in ("all","illusions"):
    assert torch.equal(rotate180(rotate180(image)),image)
    # Example prompt pairs; edit these to choose your two illusions.
    pairs=[("an oil painting of a dog","an oil painting of a snowy mountain landscape"),
           ("a watercolor of a rabbit","a watercolor of a sailing ship")]
    # If an illusion doesn't read well, change its seed here and rerun this cell.
    ANAGRAM_SEEDS=[SEED, SEED+1]
    for i,(a,b) in enumerate(pairs):
        ca, cb = model.encode(a), model.encode(b)
        def velocity(x,t):
            upright = cfg_velocity(x,t,ca,W,predict_velocity,null_condition)
            rotated = rotate180(cfg_velocity(rotate180(x),t,cb,W,predict_velocity,null_condition))
            return (upright + rotated) / 2
        x=resumable_sample(f"anagram_{i}",noise(ANAGRAM_SEEDS[i]),0,velocity,cache_key=[a,b,ANAGRAM_SEEDS[i]])
        save(f"anagram_{i}",x); panel(f"anagram_views_{i}",{a:x,b:rotate180(x)},2)
    metrics["rotation_involution_check"]=True
    metrics["anagram_prompts"] = pairs
    metrics["anagram_seeds"] = ANAGRAM_SEEDS
    save_metrics()


## 3.4 Hybrid Images

Implement `make_hybrids` using `lowpass(v_a) + v_b - lowpass(v_b)` in each Euler update.
Prompt `a` determines the distant structure; prompt `b` determines the close-up detail.
Use the same image and time for both CFG predictions. The provided low-pass helper uses
Gaussian blur with sigma 16 and kernel size 265 at 512×512.

Choose two prompt pairs and generate **two hybrid images**. Show each at full and reduced size,
labeling both prompts. The example pairs below match the subjects and viewing directions
shown on the webpage; you can choose other pairs. Several attempts may be needed.

In [ ]:
if GROUP in ("all", "illusions"):
    # (distant / low-frequency prompt, close / high-frequency prompt)
    pairs = [("a lithograph of a lion", "a lithograph of a mountain range"),
             ("a lithograph of an owl", "a lithograph of autumn leaves")]
    # If a hybrid doesn't read well, change its seed here and rerun this cell.
    HYBRID_SEEDS = [SEED, SEED + 1]
    for i, (distant_prompt, close_prompt) in enumerate(pairs):
        ca, cb = model.encode(distant_prompt), model.encode(close_prompt)
        def velocity(x,t):
            va = cfg_velocity(x,t,ca,W,predict_velocity,null_condition)
            vb = cfg_velocity(x,t,cb,W,predict_velocity,null_condition)
            return lowpass(va) + vb - lowpass(vb)
        x = resumable_sample(f"hybrid_{i}", noise(HYBRID_SEEDS[i]), 0, velocity, cache_key=[distant_prompt,close_prompt,HYBRID_SEEDS[i]])
        save(f"hybrid_{i}", x)
        panel(f"hybrid_views_{i}", {f"Near: {close_prompt}": x}, columns=1)
        print("Close:", close_prompt)
        display(to_pil(x))
        print("Distant:", distant_prompt)
        display(to_pil(x).resize((64, 64), Image.Resampling.LANCZOS))
    metrics["hybrid_lowpass_sigma"] = 16
    metrics["hybrid_prompts"] = pairs
    metrics["hybrid_seeds"] = HYBRID_SEEDS
    save_metrics()


## CS180 scope
Sections 1.1–3.4 above cover the required Part A work. Extra transformations and a course logo are CS280A-only requirements and are not included.


In [ ]:
# No additional CS280A experiments are required for this submission.


### Save your results

Inspect your saved images as well as the numerical checks. The cell below saves the run's
settings and metrics alongside its figures. Save this notebook with its outputs before
disconnecting from the runtime.

In [ ]:
metrics["peak_memory_gib"]=torch.cuda.max_memory_allocated()/2**30
(OUT/"metrics.json").write_text(json.dumps(metrics,indent=2))
print(json.dumps(metrics,indent=2))

## Export results and Part A webpage
Run the following cell after generation, or at any time to save partial progress. It exports all generated figures, metrics, and a standalone Part A report.


In [ ]:
# Build a standalone Part A webpage from this run, then download the complete package.
# This cell can also be run early to download partial progress.
import html as html_tools
import zipfile

def escape(value):
    return html_tools.escape(str(value))

expected = json.loads((ROOT / "expected_parta.json").read_text())
missing = [path for path in expected if not (ROOT / path).is_file()]
metrics["missing_required_figures"] = missing
metrics["parta_figures_complete"] = not missing
save_metrics()

def figures(names):
    tags = []
    for name in names:
        path = OUT / (name + ".png")
        if path.exists():
            tags.append(f'<figure><a href="results/partA/{name}.png"><img src="results/partA/{name}.png" alt="{escape(name)}" loading="lazy"></a><figcaption>{escape(name.replace("_", " "))}</figcaption></figure>')
    return "\n".join(tags)

sections = []
def section(title, body, names):
    sections.append(f'<section><h2>{escape(title)}</h2><p>{body}</p>{figures(names)}</section>')

section("1.1 Constructing noisy training inputs",
    "I used my photograph of a brick building in warm sunlight. Its window frames, brick texture, and carved facade provide fine details for comparing denoisers. "
    "The image is center-cropped and resized to 512 × 512 RGB, with values in [−1, 1]. A single Gaussian noise tensor (seed 180) is reused at t = 0.25, 0.50, and 0.75. "
    "The interpolation is xₜ = (1 − t)ε + tx₁: t = 0 is noise and t = 1 is the clean image.", ["training_inputs"])
blur_text = "; ".join(f"t={t}: σ={sigma:g}" for t,sigma in metrics.get("blur_sigma", {}).items())
section("1.2 Classical denoising",
    "The Gaussian baseline first divides xₜ by t to restore the clean image's amplitude, then applies a Gaussian blur. "
    "I compare σ = 1, 2, 4, 8, and 16 and choose the smallest full-image mean squared error against the clean reference before clipping. "
    "This is an oracle parameter selection for this baseline, rather than a method that assumes the clean reference is available at deployment. "
    "Blur reduces independent noise by averaging, while also smoothing real edges and texture. " + escape(blur_text),
    ["blur_0.25", "blur_0.5", "blur_0.75"])
section("1.3 One-step denoising",
    "Using the prompt “a high quality photo,” the model predicts velocity rather than noise. The endpoint estimate is x̂₁ = xₜ + (1 − t)vθ(xₜ,t,c). "
    "Each estimate below uses one model prediction and the same saved noisy tensor as the Gaussian baseline.",
    ["one_step_0.25", "one_step_0.5", "one_step_0.75"])
rms_text = "; ".join(f"t={t}: mean per-pixel RMS={value:.5f}" for t,value in metrics.get("velocity_rms_mean", {}).items())
section("1.4 Visualizing velocity",
    "The predicted update is (1 − t)vθ and the exact target update is (1 − t)(x₁ − ε). "
    "Both use the same saved image/noise pair. The error is the square root of the mean squared update difference across RGB at each pixel, computed before display clipping. "
    "All signed update panels share one symmetric scale, with zero shown as middle gray. All heatmaps share the same linear limits. " + escape(rms_text),
    ["velocity_original", "velocity_0.25", "velocity_0.50", "velocity_0.75"])
mse_text = "; ".join(f"{name}: MSE={value:.6f}" for name,value in metrics.get("reconstruction_mse", {}).items())
section("1.5 Iterative denoising with Euler sampling",
    "Euler integration updates x ← x + hv(x,t), recomputing velocity after each step. Starting at t = 0.50, h = 0.02 gives 25 updates. "
    "The trajectory shows the starting state, every fifth update, and the endpoint. The implementation shortens the last step when necessary and is checked against an analytic constant field, including t_start = 0.93. "
    "The comparison uses exactly the same t = 0.50 noisy input for blur, one-step, and Euler reconstruction. " + escape(mse_text),
    ["trajectory", "comparison"])
section("1.6 Generating images from noise",
    "Five independent Gaussian inputs, with seeds 180–184, are integrated from t = 0 to 1 using 50 Euler updates. The prompt is “a high quality photo” and guidance is disabled. "
    "The grid records the actual samples for assessing subject bias, visual quality, and diversity.", ["conditional_samples"])
section("2.1 Classifier-free guidance",
    "CFG combines the conditional and empty-prompt predictions at the same state and time: vCFG = vc + w(vc − vu), with w = 7 and h = 0.02. "
    "The five initial noises are identical to those in Section 1.6, so each seed provides a paired comparison. A numerical check verifies that w = 0 reproduces ordinary conditional prediction. "
    "Guidance strengthens the conditional direction; image quality and diversity can be compared directly in the two grids.", ["cfg_samples"])
section("3.1 Image-to-image editing",
    "Each input is interpolated with a fixed seed-180 noise tensor, then integrated to t = 1 using “a high quality photo,” w = 7, and h = 0.02. "
    "The starting times are 0.02, 0.04, 0.06, 0.10, 0.14, 0.25, and 0.50. Smaller times preserve less input information and allow larger changes. "
    "Inputs are the Campanile, my brick-building photograph, my yellow-notebook photograph, a supplied web painting, a carrot drawing, and a watercolor flower drawing.",
    ["edit_campanile", "edit_my_image_1", "edit_my_image_2", "edit_web_image", "edit_drawing_1", "edit_drawing_2"])
prompt_text = "<br>".join(f"<strong>{escape(name)}:</strong> {escape(prompt)}" for name,prompt in metrics.get("edit_prompts", {}).items())
section("3.2 Text-guided image editing",
    "The same three photographic inputs, initial noises, and seven starting times are used with new prompts. "
    "The Campanile is directed toward a coastal lighthouse, the brick facade toward a vine-covered building, and the yellow notebook toward a red leather notebook. "
    "The panels show the balance between input structure and the requested change as the starting time varies.<br>" + prompt_text,
    ["text_edit_campanile", "text_edit_my_image_1", "text_edit_my_image_2"])
anagram_text = "<br>".join(f"{escape(a)} ↔ {escape(b)}" for a,b in metrics.get("anagram_prompts", []))
section("3.3 Visual anagrams",
    "At every Euler step, I average an upright CFG prediction for prompt A with a CFG prediction for prompt B evaluated on the image rotated 180°, then rotated back into the upright frame. "
    "The rotation flips both spatial axes and applying it twice is checked to recover the input exactly. Each pair below shows both viewing orientations.<br>" + anagram_text,
    ["anagram_views_0", "anagram_views_1"])
hybrid_text = "<br>".join(f"Distant: {escape(a)}; close: {escape(b)}" for a,b in metrics.get("hybrid_prompts", []))
section("3.4 Hybrid images",
    "The Euler velocity combines the low-pass CFG prediction for the distant prompt with the high-pass CFG prediction for the close prompt: LP(va) + vb − LP(vb). "
    "The Gaussian blur has kernel size 265 and σ = 16 at 512 × 512. Full-size and reduced views are shown for inspecting the two frequency regimes.<br>" + hybrid_text,
    ["hybrid_0", "hybrid_1"])
# Add reduced views using the same image; no resampling changes the saved output.
sections[-1] += "".join(f'<figure class="distant"><img src="results/partA/hybrid_{i}.png" alt="Hybrid {i+1}, distant view"><figcaption>Hybrid {i+1}: distant view</figcaption></figure>' for i in range(2) if (OUT/f'hybrid_{i}.png').exists())
status = "All required Part A figure files are present." if not missing else f"Partial run: {len(missing)} required figures are still missing."
report = '''<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>CS180 Project 3 — Part A — Sohum Phadke</title><style>
body{font:17px/1.6 system-ui,sans-serif;color:#222;background:#faf9f6;margin:0}main{max-width:1150px;margin:auto;padding:40px 24px}h1{font-size:2.5rem;line-height:1.15}h2{border-bottom:1px solid #ccc;padding-bottom:10px}section{margin:64px 0}figure{margin:24px 0;padding:12px;background:white;border:1px solid #ddd}img{display:block;max-width:100%;height:auto;margin:auto}figcaption{font-size:.85rem;color:#555;text-align:center;margin-top:8px}.distant img{width:128px}header{border-bottom:2px solid #333;padding-bottom:24px}p{max-width:90ch}pre{white-space:pre-wrap;font-size:.8rem}
</style></head><body><main><header><p>CS180 · Fall 2026 · Sohum Phadke</p><h1>Fun with Flow Models — Part A</h1><p>Using a pretrained PixNerd flow model to compare denoisers, generate images, edit photographs and drawings, and combine text prompts into optical illusions. No model training is required.</p></header>'''
report += '<p>' + escape(status) + '</p>' + "\n".join(sections)
report += '<section><h2>Reproducibility</h2><p>Images use the supplied preprocessing and float32 network evaluation. Run settings, prompt pairs, seeds, reconstruction errors, and model revisions are recorded below. This page is generated from the current run; it does not substitute for inspecting sample quality.</p><pre>' + escape(json.dumps(metrics,indent=2)) + '</pre></section></main></body></html>'
report_path = ROOT / "partA.html"
report_path.write_text(report)
zip_path = ROOT / "parta_results.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path,"partA.html")
    for path in sorted(OUT.iterdir()):
        if path.is_file() and path.suffix.lower() in (".png", ".json", ".pt"):
            archive.write(path,"results/partA/" + path.name)
print(status)
if missing:
    print("Missing:", *missing, sep="\n  ")
print("Export:", zip_path)
print("Unzip into your local project3 folder. Open partA.html for the standalone report; the result paths also match project3.html.")
print("Review your illusions and hybrids. If changing a prompt or seed, rerun that section, then rerun this export cell.")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))


# Deliverable Checklist

- Include every section's required results and explanations on your project webpage.
- Submit your website PDF and code to the corresponding Gradescope assignments.
- The Google Form is not required for Part A; complete it after both parts are finished.
- CS280A students should also include both required Bells & Whistles tasks.

The project is due October 20, 2026. Follow the assignment webpage for complete submission instructions.